In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# 1. LOAD DATA
# =========================================================
FLOW_FILE        = "../data/output/park_flows_property_10km.csv"
PARKS_AUDIT_FILE = "../data/output/parks_audit.csv"

print("Loading flow data...")
final_df = pd.read_csv(FLOW_FILE)
final_df.columns = final_df.columns.str.strip()
print(f"Loaded {len(final_df)} flow rows.")

print("Loading parks audit data...")
parks_df = pd.read_csv(PARKS_AUDIT_FILE)
parks_df.columns = parks_df.columns.str.strip()
print(f"Loaded {len(parks_df)} park rows.")


In [ ]:
# =========================================================
# 2. VISIT DISTRIBUTION PLOT (All Properties)
# =========================================================
park_visit_totals = final_df.groupby('gis_prop_num')['visits'].sum().sort_values(ascending=True)

plt.figure(figsize=(15, 6))
plt.bar(range(len(park_visit_totals)), park_visit_totals.values, width=1.0, color='teal', edgecolor='none')
plt.title('Total Visits per NYC Park Property (Sorted Ascending)', fontsize=15)
plt.xlabel(f'Individual Properties (N={len(park_visit_totals)})', fontsize=12)
plt.ylabel('Total Cumulative Visits', fontsize=12)
plt.xticks([])
plt.tight_layout()
plt.show()


In [ ]:
# =========================================================
# 3. TOP 10 MOST VISITED PROPERTIES
# =========================================================
park_rankings = final_df.groupby(['gis_prop_num', 'property_name'])['visits'].sum().reset_index()
top_10 = park_rankings.nlargest(10, 'visits')

# Attach acres and forever_wild_id from audit
prop_attrs = (
    parks_df[['gis_prop_num', 'acres', 'forever_wild_id']]
    .drop_duplicates(subset='gis_prop_num')
)
top_10 = top_10.merge(prop_attrs, on='gis_prop_num', how='left')

top_10 = top_10[[
    'gis_prop_num',
    'property_name',
    'visits',
    'acres',
    'forever_wild_id',
]].rename(columns={'visits': 'total_visits'})

print("\n--- Top 10 Most Visited Properties ---")
print(top_10.to_string(index=False))
